In [21]:
from pathlib import Path

import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/googleplaystore.csv")
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [22]:
def data_quality_report(data):
    """Return a compact data-quality report for a DataFrame."""
    return pd.DataFrame({
        "dtype": data.dtypes.astype(str),
        "missing_count": data.isna().sum(),
        "missing_percent": (data.isna().mean() * 100).round(2),
        "unique_non_null": data.nunique(dropna=True),
    })

initial_shape = df.shape
initial_duplicate_count = int(df.duplicated().sum())
initial_missing_count = int(df.isna().sum().sum())

print("Initial rows and columns:", initial_shape)
print("Exact duplicate rows:", initial_duplicate_count)
print("Total missing cells:", initial_missing_count)

print(data_quality_report(df))


Initial rows and columns: (10841, 13)
Exact duplicate rows: 483
Total missing cells: 1487
                  dtype  missing_count  missing_percent  unique_non_null
App                 str              0             0.00             9660
Category            str              0             0.00               34
Rating          float64           1474            13.60               40
Reviews             str              0             0.00             6002
Size                str              0             0.00              462
Installs            str              0             0.00               22
Type                str              1             0.01                3
Price               str              0             0.00               93
Content Rating      str              1             0.01                6
Genres              str              0             0.00              120
Last Updated        str              0             0.00             1378
Current Ver         str           

In [23]:
duplicate_mask = df.duplicated(keep="first")
duplicate_rows = df.loc[duplicate_mask].copy()

print("Exact duplicate rows identified:", int(duplicate_mask.sum()))
print(duplicate_rows.head(10))

rows_before_deduplication = len(df)
df = df.drop_duplicates(keep="first").copy()
rows_removed_as_duplicates = rows_before_deduplication - len(df)

print("Rows removed:", rows_removed_as_duplicates)
print("Shape after duplicate removal:", df.shape)


Exact duplicate rows identified: 483
                              App  Category  Rating Reviews  \
229  Quick PDF Scanner + OCR FREE  BUSINESS     4.2   80805   
236                           Box  BUSINESS     4.2  159872   
239            Google My Business  BUSINESS     4.4   70991   
256           ZOOM Cloud Meetings  BUSINESS     4.4   31614   
261     join.me - Simple Meetings  BUSINESS     4.0    6989   
265                           Box  BUSINESS     4.2  159872   
266                      Zenefits  BUSINESS     4.2     296   
267                    Google Ads  BUSINESS     4.3   29313   
268            Google My Business  BUSINESS     4.4   70991   
269                         Slack  BUSINESS     4.4   51507   

                   Size     Installs  Type Price Content Rating    Genres  \
229  Varies with device   5,000,000+  Free     0       Everyone  Business   
236  Varies with device  10,000,000+  Free     0       Everyone  Business   
239  Varies with device   5,000,000+  

In [24]:
# Convert Rating to numeric for validation. Invalid text values become NaN.
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

invalid_rating_mask = df["Rating"].notna() & ~df["Rating"].between(1, 5)

invalid_rating_records = df.loc[
    invalid_rating_mask,
    [column for column in ["App", "Category", "Rating", "Reviews"] if column in df.columns]
].copy()

print("Invalid non-null ratings:", int(invalid_rating_mask.sum()))
print(invalid_rating_records)

df.loc[invalid_rating_mask, "Rating"] = np.nan

print("Ratings outside 1–5 remaining:",
      int((df["Rating"].notna() & ~df["Rating"].between(1, 5)).sum()))


Invalid non-null ratings: 1
                                           App Category  Rating Reviews
10472  Life Made WI-Fi Touchscreen Photo Frame      1.9    19.0    3.0M
Ratings outside 1–5 remaining: 0


In [25]:
missing_after_cleaning = (
    pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
    })
    .sort_values("missing_count", ascending=False)
)

print(missing_after_cleaning[missing_after_cleaning["missing_count"] > 0])


                missing_count  missing_percent
Rating                   1466            14.15
Current Ver                 8             0.08
Android Ver                 3             0.03
Content Rating              1             0.01
Type                        1             0.01


In [26]:
# Inspect repeated app names, if the App column exists.
if "App" in df.columns:
    repeated_app_mask = df["App"].notna() & df["App"].duplicated(keep=False)
    repeated_apps = (
        df.loc[repeated_app_mask]
        .sort_values("App")
    )
    print("Rows belonging to repeated app names:", len(repeated_apps))
    print(repeated_apps.head(20))

# Show distinct values for fields that should have a limited set of labels.
for column in ["Type", "Content Rating"]:
    if column in df.columns:
        print(f"\\n{column} values (including missing):")
        print(df[column].value_counts(dropna=False).head(20))


Rows belonging to repeated app names: 1221
                                                    App            Category  \
5415                            365Scores - Live Scores              SPORTS   
3083                            365Scores - Live Scores              SPORTS   
1755                                        8 Ball Pool                GAME   
1703                                        8 Ball Pool                GAME   
1871                                        8 Ball Pool                GAME   
1844                                        8 Ball Pool                GAME   
1675                                        8 Ball Pool                GAME   
1970                                        8 Ball Pool                GAME   
3953                                        8 Ball Pool              SPORTS   
559                     95Live -SG#1 Live Streaming App              DATING   
662                     95Live -SG#1 Live Streaming App              DATING   
4991     

In [27]:
final_shape = df.shape
final_duplicate_count = int(df.duplicated().sum())
final_missing_count = int(df.isna().sum().sum())

comparison = pd.DataFrame({
    "Before cleaning": {
        "Rows": initial_shape[0],
        "Columns": initial_shape[1],
        "Exact duplicate rows": initial_duplicate_count,
        "Missing cells": initial_missing_count,
    },
    "After cleaning": {
        "Rows": final_shape[0],
        "Columns": final_shape[1],
        "Exact duplicate rows": final_duplicate_count,
        "Missing cells": final_missing_count,
    },
})

print(comparison)
print(data_quality_report(df))


                      Before cleaning  After cleaning
Rows                            10841           10358
Columns                            13              13
Exact duplicate rows              483               0
Missing cells                    1487            1479
                  dtype  missing_count  missing_percent  unique_non_null
App                 str              0             0.00             9660
Category            str              0             0.00               34
Rating          float64           1466            14.15               39
Reviews             str              0             0.00             6002
Size                str              0             0.00              462
Installs            str              0             0.00               22
Type                str              1             0.01                3
Price               str              0             0.00               93
Content Rating      str              1             0.01                6


In [28]:
OUTPUT_PATH = Path("../data/processed/googleplaystore_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"Cleaned dataset saved to: {OUTPUT_PATH.resolve()}")
print(f"Final dataset shape: {df.shape}")


Cleaned dataset saved to: C:\Github_Projects\Google_AppStore_EDA\data\processed\googleplaystore_cleaned.csv
Final dataset shape: (10358, 13)


Operations performed in data cleaning
- Removed exact duplicate rows, retaining the first occurrence.
- Converted Rating to numeric for validation.
- Replaced ratings outside the valid 1–5 range with missing values.
- Preserved other missing values rather than applying arbitrary imputation.
- Saved the cleaned dataset without overwriting the raw source.